In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Data/processed/online_retail_cleaned.csv")
df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,total_price
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [2]:
df["invoicedate"]=pd.to_datetime(df["invoicedate"])

In [4]:
snapshot_date = df["invoicedate"].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2011-12-10 12:50:00')

In [5]:
recency_df = df.groupby("customerid")["invoicedate"].max().reset_index()
recency_df["recency"] = (snapshot_date-recency_df["invoicedate"]).dt.days
recency_df = recency_df[["customerid", "recency"]]


In [6]:
frequency_df = df.groupby("customerid")["invoiceno"].nunique().reset_index()
frequency_df.columns=["customerid", "frequency"]

In [9]:
monetary_df = df.groupby("customerid")["total_price"].sum().reset_index()
monetary_df.columns = ["customerid","monetary"]

In [10]:
rfm = recency_df.merge(frequency_df, on="customerid")
rfm = rfm.merge(monetary_df, on="customerid")
rfm.head()

,customerid,recency,frequency,monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


In [11]:
rfm["avg_order_value"] = rfm["monetary"]/rfm["frequency"]

In [12]:
items_df = df.groupby("customerid")["quantity"].sum().reset_index()
items_df.columns =["customerid","total_items"]

rfm = rfm.merge(items_df, on="customerid")

In [13]:
rfm["items_per_order"] = rfm["total_items"] / rfm["frequency"]

In [17]:
rfm_log = rfm.copy()

cols_to_log =["monetary","frequency", "total_items", "avg_order_value"]

for col in cols_to_log:
    rfm_log[col] = np.log1p(rfm_log[col])

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features = ["recency", "frequency", "monetary", "avg_order_value", "total_items", "items_per_order"]

rfm_scaled = rfm_log.copy()
rfm_scaled[features] = scaler.fit_transform(rfm_log[features])

In [19]:
rfm_scaled.head()

,customerid,recency,frequency,monetary,avg_order_value,total_items,items_per_order
0,12346,2.334574,-0.955214,3.707716,7.522807,3.818949,56.347343
1,12347,-0.905340,1.074425,1.414903,1.044160,1.332690,0.075178
2,12348,-0.175360,0.386304,0.720024,0.622312,1.297118,0.253529
3,12349,-0.735345,-0.955214,0.702287,2.449794,0.341302,0.288383
4,12350,2.174578,-0.955214,-0.614514,0.227101,-0.505607,-0.042254


In [20]:
rfm.to_csv("../Data/processed/rfm_raw.csv")
rfm_log.to_csv("../Data/processed/rfm_log.csv")
rfm_scaled.to_csv("../Data/processed/rfm_scaled.csv")